# KAGAMI EQ Core LoRA Colab Training

No Google Drive mount is required. This notebook downloads the approved EQ package from this project GitHub branch, trains on Colab GPU, then downloads `kagami_eq_core_lora.zip`.

In [ ]:
from pathlib import Path
from zipfile import ZipFile
import json
import os
import shutil
import subprocess
import sys
import urllib.request

package_url = 'https://raw.githubusercontent.com/ree84375-del/-ai-study-platform-/codex/kagami-eq-colab-notebook/custom_ai_engine/eq_core/eq_lora_package.zip'
work_dir = Path('/content/kagami_eq_core_download')
package_zip = work_dir / 'eq_lora_package.zip'
package_dir = Path('/content/eq_lora_package')
output_dir = Path('/content/kagami_eq_core_lora')
archive_base = Path('/content/kagami_eq_core_lora')
final_zip = Path('/content/kagami_eq_core_lora.zip')

if work_dir.exists():
    shutil.rmtree(work_dir)
work_dir.mkdir(parents=True)
urllib.request.urlretrieve(package_url, package_zip)
if package_zip.stat().st_size < 100000:
    raise RuntimeError(f'Downloaded package is too small: {package_zip.stat().st_size} bytes')

if package_dir.exists():
    shutil.rmtree(package_dir)
package_dir.mkdir(parents=True)
with ZipFile(package_zip) as zf:
    zf.extractall(package_dir)

manifest = json.loads((package_dir / 'eq_core_sft_manifest.json').read_text(encoding='utf-8'))
objective_gate = json.loads((package_dir / 'eq_objective_gate_report.json').read_text(encoding='utf-8'))
human_gate = json.loads((package_dir / 'eq_human_speech_gate_report.json').read_text(encoding='utf-8'))
preflight = {
    'package_bytes': package_zip.stat().st_size,
    'train_records': manifest.get('train_records'),
    'val_records': manifest.get('val_records'),
    'source_url_count': manifest.get('source_url_count'),
    'objective_gate': objective_gate.get('status'),
    'objective_hard_failure_count': objective_gate.get('hard_failure_count'),
    'human_speech_gate': human_gate.get('status'),
    'human_max_first_line_ratio': human_gate.get('max_first_line_ratio'),
}
print(json.dumps(preflight, ensure_ascii=False, indent=2))

if objective_gate.get('status') != 'PASS':
    raise RuntimeError('Objective Gate is not PASS.')
if human_gate.get('status') != 'PASS':
    raise RuntimeError('Human Speech Gate is not PASS.')

subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q', '-U',
    'transformers', 'peft', 'trl', 'bitsandbytes', 'accelerate', 'datasets', 'sentencepiece'
])

os.environ['EQ_TRAIN_PATH'] = str(package_dir / 'eq_core_sft_train.jsonl')
os.environ['EQ_OUTPUT_DIR'] = str(output_dir)
os.environ['EQ_ARCHIVE_BASE'] = str(archive_base)
os.environ.setdefault('EQ_BASE_MODEL_ID', 'Qwen/Qwen2.5-7B-Instruct')
os.environ.setdefault('EQ_MAX_STEPS', '300')
os.environ.setdefault('EQ_MAX_SEQ_LENGTH', '768')

if output_dir.exists():
    shutil.rmtree(output_dir)
if final_zip.exists():
    final_zip.unlink()

train_script = package_dir / 'train_eq_lora.py'
if not train_script.exists():
    raise FileNotFoundError(train_script)

import runpy
runpy.run_path(str(train_script), run_name='__main__')

if not final_zip.exists():
    raise FileNotFoundError(f'Expected output missing: {final_zip}')
done = {
    'status': 'KAGAMI_EQ_CORE_LORA_DONE',
    'download_file': str(final_zip),
    'output_zip_bytes': final_zip.stat().st_size,
}
print(json.dumps(done, ensure_ascii=False, indent=2))

try:
    from google.colab import files
    files.download(str(final_zip))
except Exception as exc:
    print('AUTO_DOWNLOAD_FAILED:', repr(exc))
    print('Download manually from:', final_zip)
